# Fig. 2 Generalization Sweep: Partial Quickcheck

This notebook is designed for the `nf_generalize_fig2` sweep while the full array is still running or partially blocked by maintenance.

It does three things:

1. audits which training tasks have final checkpoints,
2. audits which raw `train_full` sample files exist,
3. plots one-point/P(k)/image diagnostics for whatever samples are already available.

The full SSCD paper-style generalizability plot should still be produced by the offline script after sampling finishes:

```bash
sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_fig2_sscd.sbatch
```


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

SWEEP_NAME = 'nf_generalize_fig2'
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
CHECKPOINT_ROOT = Path('/scratch/huterer_root/huterer0/jiamingp/saved_runs') / SWEEP_NAME
SAMPLE_ROOT = PROJECT_DIR / 'results' / SWEEP_NAME / 'samples'
OUTPUT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'quickcheck'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.environ.get('NF_FIG2_SEED', 123))
SAMPLE_LABEL = os.environ.get('NF_FIG2_SAMPLE_LABEL', 'raw_train_full')
MAX_GENERATED = int(os.environ.get('NF_FIG2_MAX_GENERATED', 512))
MAX_REAL_RAW_CUBES = int(os.environ.get('NF_FIG2_MAX_REAL_RAW_CUBES', 16))
PK_NBINS = int(os.environ.get('NF_FIG2_PK_NBINS', 30))

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})

print('project:', PROJECT_DIR)
print('manifest:', MANIFEST_PATH)
print('checkpoint root:', CHECKPOINT_ROOT)
print('sample root:', SAMPLE_ROOT)
print('seed:', SEED)


In [ ]:
rows = json.loads(MANIFEST_PATH.read_text())
for task_id, row in enumerate(rows):
    row['task_id'] = task_id

manifest_df = pd.DataFrame(rows)
display(manifest_df[[
    'task_id', 'run_name', 'arch', 'dataset_tag', 'dataset_size',
    'epochs', 'steps_per_epoch', 'actual_updates', 'checkpoint_every_n_epochs',
    'n_train_simulations'
]])


## Training Checkpoint Audit

A run is treated as complete if its latest `checkpoint-epoch-*` directory is at least `epochs - 1` from the manifest. This works even if the directory name is zero-padded or not.


In [ ]:
EPOCH_RE = re.compile(r'checkpoint-epoch-(\d+)')

def checkpoint_epoch(path: Path) -> int | None:
    m = EPOCH_RE.search(path.name)
    return int(m.group(1)) if m else None

ckpt_rows = []
for row in rows:
    ckpt_dir = Path(row['checkpoint_dir'])
    ckpts = sorted(ckpt_dir.glob('checkpoint-epoch-*')) if ckpt_dir.exists() else []
    parsed = [(checkpoint_epoch(p), p) for p in ckpts]
    parsed = [(e, p) for e, p in parsed if e is not None]
    latest_epoch = max((e for e, _ in parsed), default=None)
    latest_path = next((p for e, p in parsed if e == latest_epoch), None) if latest_epoch is not None else None
    final_epoch = int(row['epochs']) - 1
    ckpt_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'expected_final_epoch': final_epoch,
        'latest_epoch': latest_epoch,
        'final_checkpoint': latest_epoch is not None and latest_epoch >= final_epoch,
        'n_checkpoints': len(parsed),
        'checkpoint_dir': str(ckpt_dir),
        'latest_checkpoint': str(latest_path) if latest_path else '',
    })

ckpt_df = pd.DataFrame(ckpt_rows).sort_values(['arch', 'dataset_size'])
display(ckpt_df)
print('checkpoint audit:', ckpt_df['final_checkpoint'].value_counts(dropna=False).to_dict())

completed_ids = ckpt_df.loc[ckpt_df['final_checkpoint'], 'task_id'].astype(int).tolist()
missing_ids = ckpt_df.loc[~ckpt_df['final_checkpoint'], 'task_id'].astype(int).tolist()
print('completed task ids:', completed_ids)
print('missing/pending task ids:', missing_ids)


## Training Loss Curves

This reads the latest `metrics_epoch_*.json` from each checkpoint directory and plots loss against optimizer updates, not just epoch number. That matters here because the small-
N runs have many more epochs but are configured to have roughly the same total optimizer-update budget.

In [ ]:
def metric_candidates(row: dict[str, Any]) -> list[Path]:
    root = Path(row['checkpoint_dir'])
    paths: list[Path] = []
    if root.exists():
        paths.extend(sorted(root.glob('metrics_epoch_*.json')))
        metrics_json = root / 'metrics.json'
        if metrics_json.exists():
            paths.append(metrics_json)
        for ckpt in sorted(root.glob('checkpoint-epoch-*')):
            paths.extend(sorted(ckpt.glob('metrics*.json')))
    return paths


def _flatten_numeric(values: Any) -> np.ndarray:
    if values is None:
        return np.asarray([], dtype=float)
    out: list[float] = []

    def visit(x: Any) -> None:
        if x is None:
            return
        if isinstance(x, dict):
            for key in ('loss', 'value', 'mean', 'avg'):
                if key in x:
                    visit(x[key])
                    return
            return
        if isinstance(x, (list, tuple, np.ndarray)):
            for item in x:
                visit(item)
            return
        try:
            out.append(float(x))
        except (TypeError, ValueError):
            return

    visit(values)
    return np.asarray(out, dtype=float)


def read_latest_metrics(row: dict[str, Any]) -> tuple[dict[str, Any], Path | None]:
    paths = metric_candidates(row)
    if not paths:
        return {}, None
    def score(path: Path) -> tuple[int, float]:
        epoch = checkpoint_epoch(path) if 'metrics_epoch_' in path.name else -1
        return (epoch if epoch is not None else -1, path.stat().st_mtime)
    latest = max(paths, key=score)
    with latest.open() as f:
        return json.load(f), latest


def downsample_xy(x: np.ndarray, y: np.ndarray, max_points: int = 1200) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) <= max_points:
        return x, y
    idx = np.linspace(0, len(x) - 1, max_points, dtype=int)
    return x[idx], y[idx]

loss_by_run: dict[str, dict[str, Any]] = {}
loss_rows = []
for row in rows:
    metrics, metrics_path = read_latest_metrics(row)
    epoch_loss = _flatten_numeric(metrics.get('epoch_loss'))
    batch_loss = _flatten_numeric(metrics.get('loss', metrics.get('batch_loss')))
    epoch_lr = _flatten_numeric(metrics.get('epoch_lr', metrics.get('lr')))
    loss_by_run[row['run_name']] = {
        'metrics': metrics,
        'metrics_path': metrics_path,
        'epoch_loss': epoch_loss,
        'batch_loss': batch_loss,
        'epoch_lr': epoch_lr,
    }
    loss_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'steps_per_epoch': int(row.get('steps_per_epoch', 1)),
        'metrics_path': str(metrics_path) if metrics_path else None,
        'n_epoch_loss': len(epoch_loss),
        'final_epoch_loss': float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
        'best_epoch_loss': float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
        'n_batch_loss': len(batch_loss),
        'final_batch_loss': float(batch_loss[-1]) if len(batch_loss) else np.nan,
    })

loss_df = pd.DataFrame(loss_rows).sort_values(['arch', 'dataset_size'])
display(loss_df)

if loss_df['n_epoch_loss'].sum() == 0 and loss_df['n_batch_loss'].sum() == 0:
    print('No training metrics JSON found yet.')
else:
    for arch, sub_rows in pd.DataFrame(rows).sort_values('dataset_size').groupby('arch'):
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
        for _, row in sub_rows.iterrows():
            run_name = row['run_name']
            info = loss_by_run.get(run_name, {})
            label = f"N={int(row['dataset_size'])}"
            steps_per_epoch = max(1, int(row.get('steps_per_epoch', 1)))
            epoch_loss = np.asarray(info.get('epoch_loss', []), dtype=float)
            if len(epoch_loss):
                x = np.arange(len(epoch_loss), dtype=float) * steps_per_epoch
                x, y = downsample_xy(x, epoch_loss)
                axes[0].plot(x, y, lw=1.4, label=label)
            batch_loss = np.asarray(info.get('batch_loss', []), dtype=float)
            if len(batch_loss):
                x = np.arange(len(batch_loss), dtype=float)
                y = batch_loss
                if len(y) > 0:
                    window = max(1, len(y) // 1200)
                    if window > 1:
                        kernel = np.ones(window, dtype=float) / window
                        y = np.convolve(y, kernel, mode='valid')
                        x = x[:len(y)] + 0.5 * (window - 1)
                x, y = downsample_xy(x, y)
                axes[1].plot(x, y, lw=1.2, alpha=0.85, label=label)
            epoch_lr = np.asarray(info.get('epoch_lr', []), dtype=float)
            if len(epoch_lr):
                x = np.arange(len(epoch_lr), dtype=float) * steps_per_epoch
                x, y = downsample_xy(x, epoch_lr)
                axes[2].plot(x, y, lw=1.2, label=label)

        axes[0].set_title('epoch loss')
        axes[0].set_xlabel('optimizer update')
        axes[0].set_ylabel('mean training loss')
        axes[1].set_title('batch loss, smoothed')
        axes[1].set_xlabel('optimizer update')
        axes[1].set_ylabel('training loss')
        axes[2].set_title('learning rate')
        axes[2].set_xlabel('optimizer update')
        axes[2].set_ylabel('LR')
        for ax in axes:
            ax.grid(alpha=0.25)
            ax.set_yscale('log')
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.10), ncol=min(5, len(labels)))
        fig.suptitle(f'{arch}: Fig.2 sweep training curves')
        fig.tight_layout(rect=(0, 0.12, 1, 0.92))
        out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_training_curves.png'
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print('wrote', out)
        plt.show()

## Sample Audit and Commands

Run sampling only for task IDs with completed final checkpoints. If maintenance blocks long training jobs, sampling should still be short, but the scheduler may still reserve nodes depending on the maintenance window.


In [ ]:
def sample_path_for(row: dict[str, Any], seed: int = SEED, sample_label: str = SAMPLE_LABEL) -> Path:
    if row.get('sample_path'):
        return PROJECT_DIR / str(row['sample_path']).format(seed=seed, sample_label=sample_label)
    return SAMPLE_ROOT / f"{row['run_name']}_seed{seed}_{sample_label}.npz"


def npz_n(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        with np.load(path) as data:
            key = 'samples' if 'samples' in data.files else data.files[0]
            return int(data[key].shape[0])
    except Exception as exc:
        print('failed reading', path, exc)
        return 0

sample_rows = []
for row in rows:
    path = sample_path_for(row)
    n = npz_n(path)
    sample_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'final_checkpoint': bool(ckpt_df.set_index('task_id').loc[row['task_id'], 'final_checkpoint']),
        'n_available': n,
        'status': 'ok' if n > 0 else 'missing',
        'sample_path': str(path),
    })

sample_df = pd.DataFrame(sample_rows).sort_values(['arch', 'dataset_size'])
display(sample_df)
print('sample audit:', sample_df['status'].value_counts().to_dict())

to_sample = sample_df[(sample_df['final_checkpoint']) & (sample_df['n_available'] == 0)]['task_id'].astype(int).tolist()
if to_sample:
    ids = ','.join(map(str, to_sample))
    print('Submit samples for completed checkpoints:')
    print(f'cd {PROJECT_DIR}')
    print(f'sbatch -A huterer0 --array={ids}%8 scripts/slurm/sample_nf_generalize_fig2_array.sbatch')
else:
    print('No completed unsampled tasks found.')


## Load Available Samples

This loads only sample files that already exist. Real data is capped with `MAX_REAL_RAW_CUBES` so the notebook stays responsive; the offline SSCD script should be used for the full-reference memorization/generalization score.


In [ ]:
def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        return as_nchw(np.asarray(data[key], dtype=np.float32))


def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, int(limit), dtype=int)
    return arr[idx].copy()

loaded = {}
load_rows = []
for row in rows:
    path = sample_path_for(row)
    if not path.exists():
        continue
    config_path = PROJECT_DIR / row['config']
    generated = evenly_limit(load_npz_array(path), MAX_GENERATED)
    raw_cap = min(int(row.get('n_train_simulations', MAX_REAL_RAW_CUBES)), MAX_REAL_RAW_CUBES)
    real = as_nchw(load_real_from_config(config_path, max_raw_samples=raw_cap))
    loaded[row['run_name']] = {
        'spec': row,
        'real': real,
        'generated': generated,
        'sample_path': path,
    }
    load_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'n_real_loaded': len(real),
        'raw_cap': raw_cap,
        'n_generated': len(generated),
        'sample_path': str(path),
    })

loaded_df = pd.DataFrame(load_rows).sort_values(['arch', 'dataset_size']) if load_rows else pd.DataFrame()
display(loaded_df)
print('loaded sample rows:', len(loaded))


## Available Sample Quality Metrics

These are quick-check metrics using capped real references. They are useful for catching broken runs or broad quality trends, not for final ranking. Lower `hist_l1` and `pk_log10_mae` are better; `std_ratio` and P(k) ratios near 1 are better.


In [ ]:
metric_rows = []
for run_name, bundle in loaded.items():
    row = bundle['spec']
    real = bundle['real']
    generated = bundle['generated']
    rh = field_histogram(real, bins=120)
    gh = field_histogram(generated, bins=120)
    edges = np.asarray(rh['bin_edges'])
    width = float(np.mean(np.diff(edges)))
    hist_l1 = float(np.sum(np.abs(np.asarray(rh['hist']) - np.asarray(gh['hist']))) * width)
    metric_rows.append({
        'task_id': row['task_id'],
        'run_name': run_name,
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'n_real': len(real),
        'n_generated': len(generated),
        'hist_l1': hist_l1,
        'generated_std': gh['std'],
        'real_std': rh['std'],
        'std_ratio': gh['std'] / max(rh['std'], 1e-30),
        **power_spectrum_summary(real, generated, nbins=PK_NBINS),
    })

metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    metrics_df = metrics_df.sort_values(['arch', 'dataset_size'])
    out = OUTPUT_DIR / 'nf_generalize_fig2_partial_metrics.csv'
    metrics_df.to_csv(out, index=False)
    print('wrote', out)
    display(metrics_df)
else:
    print('No loaded samples yet.')


In [ ]:
if len(metrics_df):
    for arch, sub in metrics_df.groupby('arch'):
        x = sub['dataset_size'].astype(float)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
        axes[0].plot(x, sub['hist_l1'], marker='o')
        axes[0].set_ylabel('one-point histogram L1')
        axes[0].set_title('one-point error')
        axes[1].plot(x, sub['pk_log10_mae'], marker='o')
        axes[1].set_ylabel('P(k) log10 MAE')
        axes[1].set_title('P(k) error')
        axes[2].plot(x, sub['std_ratio'], marker='o')
        axes[2].axhline(1.0, color='black', ls=':', lw=1)
        axes[2].set_ylabel('generated std / real std')
        axes[2].set_title('field amplitude')
        for ax in axes:
            ax.set_xscale('log', base=2)
            ax.set_xlabel('training dataset size N')
            ax.grid(alpha=0.25)
        fig.suptitle(f'{arch}: available Fig.2 quick-check metrics')
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_partial_metrics.png'
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print('wrote', out)
        plt.show()


## Detailed One-point and P(k) Comparisons

The summary table above gives scalar errors. This block shows the actual one-point histogram overlays and mean power-spectrum ratios for a small set of available dataset sizes. Use `NF_FIG2_DETAIL_TAGS=d2p06,d2p10,d2p15` before launching Jupyter if you want fewer/more panels.

In [ ]:
DETAIL_TAGS = [x.strip() for x in os.environ.get('NF_FIG2_DETAIL_TAGS', 'd2p06,d2p08,d2p10,d2p12,d2p15').split(',') if x.strip()]

if loaded:
    for arch in sorted({bundle['spec']['arch'] for bundle in loaded.values()}):
        bundles = [b for b in loaded.values() if b['spec']['arch'] == arch and b['spec'].get('dataset_tag') in DETAIL_TAGS]
        bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
        if not bundles:
            print(f'No loaded rows for {arch} and detail tags {DETAIL_TAGS}')
            continue
        n = len(bundles)
        fig, axes = plt.subplots(2, n, figsize=(max(3.4 * n, 8), 7.0), squeeze=False)
        for col, bundle in enumerate(bundles):
            row = bundle['spec']
            real = bundle['real']
            generated = bundle['generated']

            rh = field_histogram(real, bins=140)
            gh = field_histogram(generated, bins=140)
            edges = np.asarray(rh['bin_edges'])
            centers = 0.5 * (edges[:-1] + edges[1:])
            axes[0, col].plot(centers, rh['hist'], color='black', lw=1.8, label='real')
            axes[0, col].plot(centers, gh['hist'], color='tab:blue', lw=1.6, label='generated')
            axes[0, col].set_yscale('log')
            axes[0, col].set_title(f"N={row['dataset_size']} one-point")
            axes[0, col].set_xlabel('normalized field')
            if col == 0:
                axes[0, col].set_ylabel('density')
                axes[0, col].legend(frameon=True)

            pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
            pk_gen, _ = batch_power_spectra(generated, nbins=PK_NBINS)
            ratio = np.nanmean(pk_gen, axis=0) / np.clip(np.nanmean(pk_real, axis=0), 1e-30, None)
            axes[1, col].plot(kbins, ratio, marker='o', ms=3.5, lw=1.5, color='tab:blue')
            axes[1, col].axhline(1.0, color='black', ls=':', lw=1.0)
            axes[1, col].set_ylim(0, max(2.0, float(np.nanquantile(ratio, 0.98)) * 1.15 if np.isfinite(ratio).any() else 2.0))
            axes[1, col].set_title('mean P(k) ratio')
            axes[1, col].set_xlabel('k bin')
            if col == 0:
                axes[1, col].set_ylabel('generated / real')
            for ax in axes[:, col]:
                ax.grid(alpha=0.25)
        fig.suptitle(f'{arch}: detailed one-point and P(k) checks')
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_detail_hist_pk.png'
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print('wrote', out)
        plt.show()
else:
    print('No loaded samples available for detailed one-point/P(k) plots.')

## Real vs Generated Image Panels

For each available run, the top row is a capped real reference slice and the bottom row is a generated sample. This is the fastest way to see whether the low-N models are producing off-manifold artifacts, memorized-looking structures, or plausible fields.


In [ ]:
if loaded:
    for arch in sorted({bundle['spec']['arch'] for bundle in loaded.values()}):
        bundles = [b for b in loaded.values() if b['spec']['arch'] == arch]
        bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
        n = len(bundles)
        fig, axes = plt.subplots(2, n, figsize=(max(3*n, 8), 5.4), squeeze=False)
        for col, bundle in enumerate(bundles):
            row = bundle['spec']
            real = bundle['real']
            generated = bundle['generated']
            vmin = float(np.nanquantile(real, 0.005))
            vmax = float(np.nanquantile(real, 0.995))
            axes[0, col].imshow(real[0, 0], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[0, col].set_title(f"N={row['dataset_size']} real")
            axes[1, col].imshow(generated[0, 0], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[1, col].set_title('generated')
            for ax in axes[:, col]:
                ax.set_xticks([])
                ax.set_yticks([])
        fig.suptitle(f'{arch}: available real vs generated examples')
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_partial_images.png'
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print('wrote', out)
        plt.show()
else:
    print('No samples available to plot yet.')


## Generated Samples Versus Closest Loaded Training Slices

This is a quick visual copy check. For each available run it takes a generated sample, searches the loaded training-slice subset for the closest map in pixel MSE, and plots generated / closest training / absolute difference. The search is capped to keep the notebook responsive; the full SSCD job is still the serious all-reference nearest-neighbor analysis.

In [ ]:
NN_MAX_REAL = int(os.environ.get('NF_FIG2_NN_MAX_REAL', 2048))
NN_MAX_GENERATED = int(os.environ.get('NF_FIG2_NN_MAX_GENERATED', 8))
NN_DISPLAY_GENERATED_INDEX = int(os.environ.get('NF_FIG2_NN_DISPLAY_GENERATED_INDEX', 0))


def closest_training_for_generated(
    generated: np.ndarray,
    real: np.ndarray,
    max_generated: int = NN_MAX_GENERATED,
    max_real: int = NN_MAX_REAL,
    real_chunk: int = 256,
) -> pd.DataFrame:
    gen = as_nchw(generated)[:max_generated].astype(np.float32, copy=False)
    ref = as_nchw(real)[:max_real].astype(np.float32, copy=False)
    gen_flat = gen.reshape(len(gen), -1).astype(np.float32, copy=False)
    ref_flat = ref.reshape(len(ref), -1).astype(np.float32, copy=False)
    gen_norm = np.linalg.norm(gen_flat, axis=1)
    ref_norm = np.linalg.norm(ref_flat, axis=1)
    rows_out = []
    for gi, g in enumerate(gen_flat):
        best_mse = np.inf
        best_idx = -1
        for start in range(0, len(ref_flat), real_chunk):
            chunk = ref_flat[start:start + real_chunk]
            diff = chunk - g[None, :]
            mse = np.mean(diff * diff, axis=1)
            local = int(np.argmin(mse))
            if float(mse[local]) < best_mse:
                best_mse = float(mse[local])
                best_idx = start + local
        denom = max(float(gen_norm[gi] * ref_norm[best_idx]), 1e-30)
        cos = float(np.dot(g, ref_flat[best_idx]) / denom)
        rows_out.append({
            'generated_index': gi,
            'nearest_real_index': best_idx,
            'nearest_mse': best_mse,
            'nearest_rmse': float(np.sqrt(best_mse)),
            'nearest_cosine': cos,
        })
    return pd.DataFrame(rows_out)

nn_rows = []
for run_name, bundle in loaded.items():
    row = bundle['spec']
    nn = closest_training_for_generated(bundle['generated'], bundle['real'])
    for record in nn.to_dict('records'):
        nn_rows.append({
            'task_id': row['task_id'],
            'run_name': run_name,
            'arch': row['arch'],
            'dataset_size': int(row['dataset_size']),
            'n_real_searched': min(len(bundle['real']), NN_MAX_REAL),
            'n_generated_searched': min(len(bundle['generated']), NN_MAX_GENERATED),
            **record,
        })

nn_df = pd.DataFrame(nn_rows)
if len(nn_df):
    nn_df = nn_df.sort_values(['arch', 'dataset_size', 'generated_index'])
    out = OUTPUT_DIR / 'nf_generalize_fig2_partial_pixel_nn.csv'
    nn_df.to_csv(out, index=False)
    print('wrote', out)
    display(nn_df)
else:
    print('No loaded samples available for nearest-training comparison.')

if len(nn_df):
    for arch in sorted(nn_df['arch'].unique()):
        bundles = [b for b in loaded.values() if b['spec']['arch'] == arch]
        bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
        n = len(bundles)
        if n == 0:
            continue
        fig, axes = plt.subplots(3, n, figsize=(max(3.0 * n, 8), 8.2), squeeze=False)
        for col, bundle in enumerate(bundles):
            row = bundle['spec']
            sub = nn_df[(nn_df['run_name'] == row['run_name']) & (nn_df['generated_index'] == NN_DISPLAY_GENERATED_INDEX)]
            if len(sub) == 0:
                sub = nn_df[nn_df['run_name'] == row['run_name']].head(1)
            if len(sub) == 0:
                for ax in axes[:, col]:
                    ax.axis('off')
                continue
            rec = sub.iloc[0]
            gi = int(rec['generated_index'])
            ri = int(rec['nearest_real_index'])
            gen_img = bundle['generated'][gi, 0]
            real_img = bundle['real'][ri, 0]
            diff = np.abs(gen_img - real_img)
            vmin = float(np.nanquantile(bundle['real'], 0.005))
            vmax = float(np.nanquantile(bundle['real'], 0.995))
            axes[0, col].imshow(gen_img, cmap='viridis', vmin=vmin, vmax=vmax)
            axes[0, col].set_title(f"N={row['dataset_size']} gen {gi}")
            axes[1, col].imshow(real_img, cmap='viridis', vmin=vmin, vmax=vmax)
            axes[1, col].set_title(f"closest train {ri}")
            axes[2, col].imshow(diff, cmap='magma')
            axes[2, col].set_title(f"MSE={rec['nearest_mse']:.3g}\ncos={rec['nearest_cosine']:.3f}")
            for ax in axes[:, col]:
                ax.set_xticks([])
                ax.set_yticks([])
        fig.suptitle(f'{arch}: generated sample versus closest loaded training slice')
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_generated_vs_pixel_nn.png'
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print('wrote', out)
        plt.show()

## Paper Fig. 2 Style Reproducibility and Generalizability

This section reproduces the structure of Fig. 2 from the paper using the runs currently available. The key statistic is computed per generated sample:

\[
s_j = \max_i \mathrm{sim}(x_j, y_i), \qquad \mathrm{GL}(\tau)=1-\Pr[s_j > \tau].
\]

For reproducibility, we compare paired generated samples from `u64` and `u128` at the same training-set size and plot the fraction above the same threshold. The fixed-`tau` plot uses a single hand-chosen threshold. The adaptive reproducibility plot uses the train-real leave-one-out q95/q99 threshold at each `N`, so the bar moves with the density of the training set. PCA is a diagnostic feature space; SSCD is the paper-style copy-detection feature space.


In [ ]:
TABLE_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'tables'
PCA_METRICS_PATH = TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_metrics.csv'
PCA_RP_PATH = TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_reproducibility.csv'
SSCD_METRICS_PATH = TABLE_DIR / 'nf_generalize_fig2_sscd_full_nn_metrics.csv'
SSCD_RP_PATH = TABLE_DIR / 'nf_generalize_fig2_sscd_full_nn_reproducibility.csv'
FIG2_PRIMARY_TAU = float(os.environ.get('NF_FIG2_PRIMARY_TAU', 0.9))
FIG2_TAU_SUFFIX = f"{FIG2_PRIMARY_TAU:.3f}".rstrip('0').rstrip('.').replace('.', 'p')


def read_table(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        if 'dataset_size' in df:
            df = df.sort_values(['arch', 'dataset_size'] if 'arch' in df else ['dataset_size'])
        return df
    return pd.DataFrame()


def plot_fig2_style(metrics: pd.DataFrame, rp: pd.DataFrame, *, feature_name: str, tau_suffix: str = FIG2_TAU_SUFFIX) -> None:
    if metrics.empty:
        print(f'Missing {feature_name} metrics table.')
        return
    gl_col = f'gen_gl_fixed_{tau_suffix}'
    val_gl_col = f'val_gl_fixed_{tau_suffix}'
    rp_col = f'rp_fixed_{tau_suffix}'
    if gl_col not in metrics.columns:
        print(f'{feature_name}: missing {gl_col}; available GL columns:', [c for c in metrics.columns if c.startswith('gen_gl_fixed_')])
        return

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.0), sharex=True)

    if not rp.empty and rp_col in rp.columns:
        for pair, sub in rp.groupby('arch_pair', sort=False):
            sub = sub.sort_values('dataset_size')
            axes[0].plot(sub['dataset_size'], sub[rp_col], 'o-', lw=2.4, label=pair)
        axes[0].set_title(f'{feature_name}: reproducibility, tau={FIG2_PRIMARY_TAU:g}')
        axes[0].set_ylabel('RP = fraction paired samples above tau')
    else:
        axes[0].text(0.5, 0.5, 'reproducibility table missing', ha='center', va='center', transform=axes[0].transAxes)
        axes[0].set_title(f'{feature_name}: reproducibility')
        axes[0].set_ylabel('RP')

    for arch, sub in metrics.groupby('arch', sort=False):
        sub = sub.sort_values('dataset_size')
        axes[1].plot(sub['dataset_size'], sub[gl_col], 'o-', lw=2.4, label=f'{arch} generated')
        if val_gl_col in sub.columns:
            axes[1].plot(sub['dataset_size'], sub[val_gl_col], 'o--', alpha=0.55, label=f'{arch} held-out real')
    axes[1].set_title(f'{feature_name}: generalizability, tau={FIG2_PRIMARY_TAU:g}')
    axes[1].set_ylabel('GL = 1 - fraction above tau')

    for ax in axes:
        ax.set_xscale('log', base=2)
        ax.set_xlabel('training dataset size N')
        ax.set_ylim(-0.03, 1.03)
        ax.grid(alpha=0.25)
        ax.legend(frameon=True)
    fig.suptitle(f'Attempted Fig. 2 reproduction using {feature_name} features')
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_paper_fig2_attempt.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()


def plot_adaptive_rp(rp: pd.DataFrame, *, feature_name: str) -> None:
    if rp.empty:
        return
    cols = [c for c in ('rp_q95', 'rp_q99') if c in rp.columns]
    if not cols:
        print(
            f'{feature_name}: adaptive reproducibility columns missing. '
            'Rerun the PCA/SSCD analyzer so the reproducibility CSV includes rp_q95/rp_q99.'
        )
        return
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for pair, sub in rp.groupby('arch_pair', sort=False):
        sub = sub.sort_values('dataset_size')
        for col in cols:
            ax.plot(sub['dataset_size'], sub[col], marker='o', lw=2.2, label=f'{pair} {col}')
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel('training dataset size N')
    ax.set_ylabel('RP = fraction paired samples above adaptive train-real threshold')
    ax.set_title(f'{feature_name}: adaptive train-real reproducibility check')
    ax.grid(alpha=0.25)
    ax.legend(frameon=True, fontsize=8)
    fig.tight_layout()
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_adaptive_rp.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()


def plot_adaptive_gl(metrics: pd.DataFrame, *, feature_name: str) -> None:
    if metrics.empty:
        return
    cols = [c for c in ('gen_gl_q95', 'gen_gl_q99', 'val_gl_q95', 'val_gl_q99') if c in metrics.columns]
    if not cols:
        return
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for arch, sub in metrics.groupby('arch', sort=False):
        sub = sub.sort_values('dataset_size')
        for col in cols:
            ls = '--' if col.startswith('val_') else '-'
            alpha = 0.55 if col.startswith('val_') else 1.0
            ax.plot(sub['dataset_size'], sub[col], marker='o', ls=ls, alpha=alpha, label=f'{arch} {col}')
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel('training dataset size N')
    ax.set_ylabel('GL = 1 - fraction above adaptive train-real threshold')
    ax.set_title(f'{feature_name}: adaptive train-real generalizability check')
    ax.grid(alpha=0.25)
    ax.legend(frameon=True, ncol=2, fontsize=8)
    fig.tight_layout()
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_adaptive_gl.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()

pca_fig2_df = read_table(PCA_METRICS_PATH)
pca_rp_df = read_table(PCA_RP_PATH)
sscd_fig2_df = read_table(SSCD_METRICS_PATH)
sscd_rp_df = read_table(SSCD_RP_PATH)

for name, df, path in [
    ('PCA metrics', pca_fig2_df, PCA_METRICS_PATH),
    ('PCA reproducibility', pca_rp_df, PCA_RP_PATH),
    ('SSCD metrics', sscd_fig2_df, SSCD_METRICS_PATH),
    ('SSCD reproducibility', sscd_rp_df, SSCD_RP_PATH),
]:
    print(name, 'loaded' if len(df) else 'missing', path)
    if len(df):
        display(df)

plot_fig2_style(pca_fig2_df, pca_rp_df, feature_name='PCA')
plot_adaptive_rp(pca_rp_df, feature_name='PCA')
plot_adaptive_gl(pca_fig2_df, feature_name='PCA')
plot_fig2_style(sscd_fig2_df, sscd_rp_df, feature_name='SSCD')
plot_adaptive_rp(sscd_rp_df, feature_name='SSCD')
plot_adaptive_gl(sscd_fig2_df, feature_name='SSCD')


## Unpaired U64-U128 Reproducibility Check

The Fig. 2 reproducibility panel above is index-paired: `u128[j]` is compared only with `u64[j]`. This section checks whether the two generated sample sets overlap without assuming the same sample order. For each generated sample from one architecture, it finds the nearest generated sample from the other architecture:

\[
s_j^{u128\to u64}=\max_i \mathrm{sim}(x^{u128}_j, x^{u64}_i).
\]

The plotted score is the fraction of these nearest-neighbor similarities above the threshold. This keeps the old paired result and adds the unpaired nearest-neighbor result as a separate diagnostic.


In [ ]:
def plot_unpaired_reproducibility(rp: pd.DataFrame, *, feature_name: str, tau_suffix: str = FIG2_TAU_SUFFIX) -> None:
    if rp.empty:
        print(f'Missing {feature_name} reproducibility table.')
        return
    paired_col = f'rp_fixed_{tau_suffix}'
    sym_col = f'rp_unpaired_fixed_{tau_suffix}'
    a_to_b_col = f'rp_unpaired_a_to_b_fixed_{tau_suffix}'
    b_to_a_col = f'rp_unpaired_b_to_a_fixed_{tau_suffix}'
    needed = [paired_col, a_to_b_col]
    missing = [c for c in needed if c not in rp.columns]
    if missing:
        print(
            f'{feature_name}: missing unpaired columns {missing}. '
            'Rerun the PCA/SSCD analyzer after pulling the latest code.'
        )
        return

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.0), sharex=True)
    display_cols = [
        'dataset_size',
        'paired_similarity_median',
        'a_to_b_nn_median',
        'b_to_a_nn_median',
        paired_col,
        a_to_b_col,
    ]
    if sym_col in rp.columns:
        display_cols.append(sym_col)
    if b_to_a_col in rp.columns:
        display_cols.append(b_to_a_col)

    for pair, sub in rp.groupby('arch_pair', sort=False):
        sub = sub.sort_values('dataset_size')
        arch_a = str(sub['arch_a'].iloc[0])
        arch_b = str(sub['arch_b'].iloc[0])
        x = sub['dataset_size']
        axes[0].plot(x, sub['paired_similarity_median'], 'o-', lw=2.2, label=f'{pair} paired median')
        axes[0].plot(x, sub['a_to_b_nn_median'], 's-', lw=2.2, label=f'{arch_a} -> nearest {arch_b}')
        if 'b_to_a_nn_median' in sub.columns:
            axes[0].plot(x, sub['b_to_a_nn_median'], '^-', lw=2.0, alpha=0.8, label=f'{arch_b} -> nearest {arch_a}')

        axes[1].plot(x, sub[paired_col], 'o-', lw=2.2, label=f'{pair} paired')
        axes[1].plot(x, sub[a_to_b_col], 's-', lw=2.2, label=f'{arch_a} -> nearest {arch_b}')
        if b_to_a_col in sub.columns:
            axes[1].plot(x, sub[b_to_a_col], '^-', lw=2.0, alpha=0.8, label=f'{arch_b} -> nearest {arch_a}')
        if sym_col in sub.columns:
            axes[1].plot(x, sub[sym_col], 'k--', lw=1.5, alpha=0.6, label='two-direction mean')

        show_cols = [c for c in display_cols if c in sub.columns]
        print(f'\n{feature_name} unpaired reproducibility columns for {pair}:')
        display(sub[show_cols])

    axes[0].set_ylabel('generated-to-generated similarity')
    axes[0].set_title(f'{feature_name}: paired vs unpaired NN similarity')
    axes[1].set_ylabel(f'RP = fraction above tau={FIG2_PRIMARY_TAU:g}')
    axes[1].set_title(f'{feature_name}: paired vs unpaired RP')
    for ax in axes:
        ax.set_xscale('log', base=2)
        ax.set_xlabel('training dataset size N')
        ax.grid(alpha=0.25)
        ax.legend(frameon=True, fontsize=8)
    axes[1].set_ylim(-0.03, 1.03)
    fig.tight_layout()
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_unpaired_reproducibility.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()


plot_unpaired_reproducibility(pca_rp_df, feature_name='PCA')
plot_unpaired_reproducibility(sscd_rp_df, feature_name='SSCD')


## Generalizability Nearest-Training Check

This is the generalizability analogue of the paired/unpaired reproducibility diagnostic. The left panel shows the typical nearest-training similarity for generated samples and held-out real samples. The right panel shows the fixed-threshold generalizability score:

\[
\mathrm{GL}(\tau)=1-\Pr[\max_i \mathrm{sim}(x_j, x_i^{\mathrm{train}})>\tau].
\]

So high nearest-training similarity pushes GL down, while held-out real gives the finite-reference baseline for how often real samples look close to the training set.


In [ ]:
def plot_generalizability_nn(metrics: pd.DataFrame, *, feature_name: str, tau_suffix: str = FIG2_TAU_SUFFIX) -> None:
    if metrics.empty:
        print(f'Missing {feature_name} metrics table.')
        return
    gen_gl_col = f'gen_gl_fixed_{tau_suffix}'
    val_gl_col = f'val_gl_fixed_{tau_suffix}'
    gen_copy_col = f'gen_copy_fraction_fixed_{tau_suffix}'
    val_copy_col = f'val_copy_fraction_fixed_{tau_suffix}'
    needed = ['gen_nn_median', gen_gl_col]
    missing = [c for c in needed if c not in metrics.columns]
    if missing:
        print(
            f'{feature_name}: missing generalizability columns {missing}. '
            'Rerun the PCA/SSCD analyzer after pulling the latest code.'
        )
        return

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.0), sharex=True)
    display_cols = [
        'arch',
        'dataset_size',
        'gen_nn_median',
        'val_nn_median',
        'gen_nn_q90',
        'val_nn_q90',
        gen_gl_col,
        val_gl_col,
        gen_copy_col,
        val_copy_col,
    ]

    for arch, sub in metrics.groupby('arch', sort=False):
        sub = sub.sort_values('dataset_size')
        x = sub['dataset_size']
        axes[0].plot(x, sub['gen_nn_median'], 'o-', lw=2.2, label=f'{arch} generated -> nearest train')
        if 'val_nn_median' in sub.columns:
            axes[0].plot(x, sub['val_nn_median'], 'o--', lw=2.0, alpha=0.65, label=f'{arch} held-out real -> nearest train')

        axes[1].plot(x, sub[gen_gl_col], 'o-', lw=2.2, label=f'{arch} generated GL')
        if val_gl_col in sub.columns:
            axes[1].plot(x, sub[val_gl_col], 'o--', lw=2.0, alpha=0.65, label=f'{arch} held-out real GL')

        show_cols = [c for c in display_cols if c in sub.columns]
        print(f'\n{feature_name} generalizability columns for {arch}:')
        display(sub[show_cols])

    axes[0].set_ylabel('nearest-training similarity median')
    axes[0].set_title(f'{feature_name}: generated/held-out vs nearest training')
    axes[1].set_ylabel(f'GL = 1 - fraction above tau={FIG2_PRIMARY_TAU:g}')
    axes[1].set_title(f'{feature_name}: fixed-threshold generalizability')
    for ax in axes:
        ax.set_xscale('log', base=2)
        ax.set_xlabel('training dataset size N')
        ax.grid(alpha=0.25)
        ax.legend(frameon=True, fontsize=8)
    axes[1].set_ylim(-0.03, 1.03)
    fig.tight_layout()
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_generalizability_nn.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()


plot_generalizability_nn(pca_fig2_df, feature_name='PCA')
plot_generalizability_nn(sscd_fig2_df, feature_name='SSCD')


## Offline Analyzer Commands

If the PCA/SSCD tables above are missing or stale, submit these jobs. They both skip missing samples, so with the current state they should draw a complete `u64` line and a partial `u128` line through `N=8192`. After tasks `18,19` finish and are sampled, rerun the same commands to fill in the last two `u128` points.

```bash
cd /home/jiamingp/diffusion_models_repo
sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_fig2_pca.sbatch
sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_fig2_sscd.sbatch
```

Expected table outputs:

```text
results/nf_generalize_fig2/tables/nf_generalize_fig2_pca_full_nn_metrics.csv
results/nf_generalize_fig2/tables/nf_generalize_fig2_pca_full_nn_reproducibility.csv
results/nf_generalize_fig2/tables/nf_generalize_fig2_sscd_full_nn_metrics.csv
results/nf_generalize_fig2/tables/nf_generalize_fig2_sscd_full_nn_reproducibility.csv
```